# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.


ideas:

responde las dudas. 
audio. te lee lo que responde (hacer que responda por pasos sin resumen y mas concisamente)
imagen: que haga un pequeño draft de lo que esta haciendo el codigo en local

tools: si preguntas por un curso de udemy donde aprender sobre codigo de ese esilo te responda con el mejor curso de ed. habria que meterle un dicc con los cursos y para que sirve cada uno.

extra: lo de que pueda cambiar de model: ollama/chat


In [15]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openai_model = "gpt-4.1-mini"
ollama_model="llama3.1" 
ollama_base_url = "http://localhost:11434/v1"



UDEMY_CATALOG = [
    {
        "title": "Ed Donner's AI Automation in n8n: Build Agents & Voice Agents on Udemy",
        "best_for": "Building low-code AI agents and voice agents using n8n workflows and automation tools."
    },
    {
        "title": "Ed Donner's AI Engineer Production Track: Deploy LLMs & Agents at Scale on Udemy",
        "best_for": "Deploying LLMs and AI agents to production with MLOps practices and cloud platforms (AWS, GCP, Azure, Vercel)."
    },
    {
        "title": "Ed Donner's Complete AI Agent Engineering Course (2025) on Udemy",
        "best_for": "Learning how to design and build AI agents end-to-end with practical projects."
    },
    {
        "title": "Ed Donner's AI Leadership Track: Gen AI & Agentic AI for Business Leaders on Udemy",
        "best_for": "Understanding GenAI and Agentic AI from a strategic and business leadership perspective."
    },
    {
        "title": "Ed Donner's AI Engineer Agentic Track: The Complete Agent & MCP Course on Udemy",
        "best_for": "Mastering agentic systems, MCP, and tool-based architectures through hands-on implementation."
    },
    {
        "title": "Ed Donner's AI Engineer Core Track: LLM Engineering, RAG, QLoRA, Agents on Udemy",
        "best_for": "Building strong foundations in LLM engineering, including RAG, fine-tuning (QLoRA), and agent systems."
    }
]


system_course_prompt = """
You are an AI course recommendation engine.

Your task:
- The user will provide their learning purpose or goal.
- You will receive a fixed catalog of Udemy courses (each with "title" and "best_for").
- You must recommend exactly ONE course from the catalog.
-When writing the course's name put it in bold letters so that it is more visual

Rules:
1. You MUST choose only from the provided catalog.
2. Do NOT invent courses.
3. Base your decision strictly on how well the user's goal matches the "best_for" description.
4. Choose the course that most directly aligns with the user's stated purpose.
5. If multiple seem relevant, pick the one that is the most specific and practical match.
6. If the goal is vague, choose the most foundational or broadly applicable course.
"""

def course_recomendation(purpose, provider):
    
    messages = [
        {"role": "system", "content": system_course_prompt},
        {"role": "user", "content": f"PURPOSE:\n{purpose}\n\nCATALOG:\n{UDEMY_CATALOG}"}
    ]

    if provider == "OpenAI":
        client = OpenAI()
        model = openai_model
    else:
        client = OpenAI(base_url=ollama_base_url, api_key="ollama")
        model = ollama_model

    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content


system_message = """
You are an expert programming assistant.

Your main role:
- Explain code clearly and step by step.
- If the user pastes code, explain what it does and why.
- If there is an error, identify the likely cause and suggest a fix.
- Be concise but precise.

Course policy:
- If course_mode == "Yes", after explaining the code you MUST call the function "get_udemy_course" with purpose equal to the user's message, then append the returned recommendation.
- Do NOT recommend courses yourself without calling the tool.
"""



course_function = {
    "name": "get_udemy_course",
    "description": "Recommend exactly one Udemy course from the internal catalog based on the user's learning purpose.",
    "parameters": {
        "type": "object",
        "properties": {
            "purpose": {
                "type": "string",
                "description": "The user's learning goal or purpose for taking a course."
            }
        },
        "required": ["purpose"],
        "additionalProperties": False
    }
}


tools = [{"type": "function", "function": course_function}]

def handle_tool_calls_ex(message, provider, code):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_udemy_course":
            arguments = json.loads(tool_call.function.arguments)
            purpose = f"I want to learn the skills needed to understand and build code like this:\n{code}"
            result_text = course_recomendation(purpose, provider=provider)
            responses.append({
                "role": "tool",
                "content": result_text,
                "tool_call_id": tool_call.id
            })
    return responses


import base64
from io import BytesIO
from PIL import Image

client_openai = OpenAI()


def artist(code):
    image_response = client_openai.images.generate(
            model="dall-e-3",
           prompt=f"""
Create a clean technical diagram that visually explains what this code does.
Show input → process → output flow.
Use minimal style, white background, simple arrows and labeled boxes.

CODE:
{code}
"""
,
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

def talker(message):
    response = client_openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

def chat(history, provider, course_mode):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message + f"\n\ncourse_mode={course_mode}"}] + history

    if provider == "OpenAI":
        client = OpenAI()
        model = openai_model
    else:
        client = OpenAI(base_url=ollama_base_url, api_key="ollama")
        model = ollama_model
 
    code = next(m["content"] for m in reversed(history) if m["role"] == "user")
        
    response = client.chat.completions.create(model=model, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls_ex(message, provider, code)
        messages.append(message)
        messages.extend(responses)
        response = client.chat.completions.create(model=model, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)
    image=artist(code)
    
    return history, voice, image

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    provider = gr.Radio(
    choices=["OpenAI", "Ollama"],
    value="OpenAI",
    label="Provider"
)
    course_mode = gr.Radio(
    choices=["No", "Yes"],
    value="No",
    label="Course recommendation?"
)
    with gr.Row():
        message = gr.Textbox(label="Enter the code you want to be explained:")
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)

# Hooking up events to callbacks

    message.submit(
    put_message_in_chatbot,
    inputs=[message, chatbot],
    outputs=[message, chatbot]
).then(
    chat,
    inputs=[chatbot, provider, course_mode],
    outputs=[chatbot, audio_output, image_output]
)
ui.launch(inbrowser=True, auth=("gonzalo", "hola"))

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.
